<a href="https://colab.research.google.com/github/voraciousnerd/iqm-quantum-school-labs/blob/main/QS26_Day3_Lab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **From abstract code to physical pulses**

# Setup

Install the appropriate packages for Python by running the following cell:

In [ ]:
%%capture
!pip install --upgrade pip
!pip install --force-reinstall --no-cache-dir iqm-pulla[notebook-pin-all]
!pip install --force-reinstall --no-cache-dir "numpy==2.3.4" "scipy>=1.14,<1.17"

**After installation, restart the session (go to Runtime -> Restart session)**

# Part 3: Readout distributions - measuring qubits

Measuring the state of a qubit is the fundamental operation we need to perform to know anything about our experiments on a quantum computer.

In this notebook, you will ...
* ... understand how the measurement of qubits works in a superconducting quantum computer.

By the end of this notebook, you will not only have a basic understanding of how to execute pulse schedules, but you will also have a feeling for the outcome to expect in your experiments.

In order to get started, make sure you have the appropriate packages installed:

## 1. Preparation

### 1.1 Connecting to the QPU station control
In the following, we will use the PulLa (Pulse Level Access) package. You can find the documentation [here](https://docs.iqm.tech/4.6/iqm-pulla/index.html
).

As a first step, we need to create a **PulLa object**. Conceptually, this is an IQM quantum computer client for connecting to the IQM server and constructing a circuit-to-pulse compiler. A compiler object defines the specific circuit-to-pulse compilation logic; it contains information about the quantum computer, like chip topology, the set of available native operations and other details that can be found [here](https://docs.iqm.tech/4.6/iqm-pulla/Configuration%20and%20Usage.html).

By default,  the `get_standard_compiler()` function fetches the default calibration set from the server.

Make sure you have the correct url and token and run the cell below:

In [ ]:
from iqm.pulla.pulla import Pulla

p = Pulla("https://resonance.iqm.tech", quantum_computer="garnet", token=input("Enter your Resonance API token: "))

compiler = p.get_standard_compiler()

## 2. Measurement operation

Let's start with the classical analogy to set the context: we have seen in the past lectures that a classical bit of information can be 0 or 1. But how do we decide whether the bit is one of the two values? We need to measure it! In a classical computer, a bit is usually physically stored as an **electrical voltage**: a voltage above a certain **threshold** represents a 1, and a voltage below that threshold represents a 0. The value of the threshold depends on the specific device.

In the case of a superconducting qubit, readout is done via **superconducting resonators** (e.g. LC oscillators) that are coupled to the transmon qubits in the QPU. The coupling between the qubit and the resonator can be described with the [Jaynes-Cummings model](https://en.wikipedia.org/wiki/Jaynes%E2%80%93Cummings_model), which allows for two different regimes of the system, the **resonant** and the **dispersive** one. In the latter, looking at the spectrum of the resonator, one can infer the state of the qubit thanks to the **shift** in the resonator's frequency caused by the coupling. The raw measurement signal is typically represented by a **complex number** as function of time. The measurement instrument integrates the signal over time to yield a complex number, one per measurement operation.

What we are used to see as a result of a measurement for our quantum circuit looks different though, right? The complex number is the quantity we need to threshold, as the voltage in the classical case! A last extra step is required: the complex number is rotated so that the difference between the states is maximal along the real axis. We then set a **calibrated threshold value** to compare with the real value of the signal, from which we finally get the well-known 0/1 labels :)

In the following, **we will not perform the thresholding** to investigate and get familiar with the raw results of a measurement operation.

### 2.1 Measuring a qubit in different states

Knowing now that the readout signal depends on the state of the qubit, our strategy will be to prepare the qubits in different states and compare observations!
We will inspect the results of 3 circuits:
1. Prepare a qubit in the $|0\rangle$ state and measure.
2. Prepare a qubit in the $|1\rangle$ state and measure.
3. Prepare a qubit in the superposition of $|0\rangle$ and $|1\rangle$ and measure.

Below we use the IQM Pulse syntax to define the circuits (you can find the documentation [here](https://docs.meetiqm.com/iqm-pulse/)). We will give a name to the circuits and append the necessary `CircuitOperation` to prepare the states above.

The PRX gate is defined as $R_{\phi}(\theta) = e^{-i(X cos(\phi)+Y sin(\phi))\frac{\theta}{2}}$. We can prepare the qubit in the three different states above changing the rotation angle $\theta$ and setting the phase angle $\phi$ to zero.

We also include the measurement operation in each circuit; `measure_fidelity` is the destructive version of the regular `measure` operation, maximizing the readout fidelity and it is used at the end of the circuit.

In [ ]:
from iqm.pulse import Circuit
from iqm.pulse import CircuitOperation as Op
import numpy as np

qubit = "QB1"
circuits = []
for name, angle in zip(["state0", "state1", "superposition"], ): #TODO in the empty list, fill in the theta angles for these three gates
    circuit = Circuit(name, [
        Op("prx", (qubit,), args={"angle": angle, "phase": 0.0}),
        Op("measure_fidelity", (qubit,), args={"key": "M"})
    ])
    circuits.append(circuit)

To see the unthresholded signals, we need to tweak the **calibration settings** of the `measure_fidelity` operation. We can see the default settings with the `get_settings` method:

In [ ]:
settings = compiler.get_settings(circuits)
settings

If you navigate through the gates settings above, you will find that for the `constant` implementation of the `measure_fidelity` operation, the **readout type** is controlled by the `acquisition_type` parameter, which is set to `threshold` by default.

We can use `settings.get_gate_node_for_locus` to drill down to the node that determines the settings of the operation. In this way, we can then change the `acquisition_type` of the implementation from `threshold` to `complex`.

Note: On Star-topology systems, the electronics require that *all* of the qubits have the same `acquisition_type`.

In [ ]:
# IQM Crystal way
settings.get_gate_node_for_locus("measure_fidelity", qubit).acquisition_type = "complex"

# IQM Star way
# for q in compiler.chip_topology.qubits:
#     settings.get_gate_node_for_locus("measure_fidelity", q).acquisition_type = "complex"

Now we can compile and execute the circuits with the modified settings with the `submit_playlist` method:

In [ ]:
job_definition, context = compiler.compile(circuits, settings=settings)
job = p.submit_playlist(job_definition, context=context)
job.wait_for_completion()

**Below you can retrieve your job ID!**

In [ ]:
job.data.id

## 3. Analyze the results

Now that the 'playlist' of instructions above has successfully run, we can extract the results and compare them visualizing their real and imaginary part.
The result dataset has many data arrays, but we are interested in the single shot results of the measurement key `M`. The key was defined in the circuit.

In [ ]:
results = job.result(compiler).dataset[f"{qubit}__M_readout_single_shot"]

We start with the first two state preparations in the $|0\rangle$ and $|1\rangle$ states:

In [ ]:
import matplotlib.pyplot as plt

state_0_results = results.sel(circuit_index=0)
state_1_results = results.sel(circuit_index=1)
plt.figure()
plt.scatter(np.real(state_0_results), np.imag(state_0_results), label="Prepare 0", s=4)
plt.scatter(np.real(state_1_results), np.imag(state_1_results), label="Prepare 1", s=4)
plt.xlabel('Re')
plt.ylabel('Im')
plt.gca().set_aspect('equal')
plt.grid()
plt.legend();

Time to stop and think! Let's make some important observations about these results:

- When preparing state $|0\rangle$ $\left(|1\rangle\right)$, most of the shots are **clustered** on the left (right). The signal has been calibrated so that the difference is maximized along the real axis.
- However, some shots are in the wrong cluster: This is due to various state preparation, gate, and measurement errors.

As we have said already above, to convert a shot to a 0 or 1 label, the system would compare the real part of the measurement shot to a threshold `t` located somewhere in the middle of the two clusters.
If `Re(shot) < t`, the result is labeled as 0, and 1 if `Re(shot) > t`.

Curious to know what the the calibrated `t` was? Run the code below!

In [ ]:
t_cal = settings.get_gate_node_for_locus("measure_fidelity", [qubit]).integration_threshold.value
print(t_cal)

**TASK: What about the superposition state?**
> Print the result below and make your observations.

In [ ]:
superposition_results = #TODO

plt.figure()
plt.scatter(np.real(superposition_results), np.imag(superposition_results), label="Superposition", s=4)
plt.xlabel('Re')
plt.ylabel('Im')
plt.gca().set_aspect('equal')
plt.grid()
plt.legend();

We observe that the clusters are in the same locations as for 0 and 1. Moreover, there is no "continuum" of shots between the two clusters, apart from few ones due to noise.

## 4. Let's play with calibration!
Now that we have seen what the typical behaviour should look like for your readout experiments, we can have some fun and tweek a couple of calibration parameters to see how the results change.

This is interesting to understand the effects of bad calibration in fundamental operations like the measurement one.  

#### 4.1 Change the threshold
Change `acquisition_type` back to `threshold`, choose a new `integration_threshold` changing the measure operation settings and then repeat the experiment.

In [ ]:
settings = compiler.get_settings(circuits) # Get new settings to reset them
measure_settings = settings.get_gate_node_for_locus("measure_fidelity", [qubit])
measure_settings.integration_threshold = # Set a new threshold starting from t_cal above
measure_settings.acquisition_type = "threshold"

In [ ]:
job_definition, context = compiler.compile(circuits, settings=settings)
job = p.submit_playlist(job_definition, context=context)
job.wait_for_completion()

What happens to the superposition state?

In [ ]:
from collections import Counter
counts = Counter(np.array(job.result().circuit_measurement_results[2]["M"]).squeeze())
print("0:", counts[0])
print("1:", counts[1])

We see that the distribution of counts is not 50-50 anymore because we misinterpret the raw signal and assign wrong 0/1 labels!

#### 4.2 Change the drive amplitude

We conclude this notebook looking at one last parameter: the amplitude of the microwave drive sent to the superconducting resonator. Let's see what is the current calibrated value below:

In [ ]:
calibrated_amplitude = settings.gates.measure_fidelity.constant.QB1.amplitude_i.value
print(calibrated_amplitude)

Change the parameter `"amplitude_i"` of the `measure_fidelity` gate with a smaller and a bigger value and repeat the experiment.

What happens for the $|0\rangle$ and $|1\rangle$ states?

In [ ]:
settings = compiler.get_settings(circuits) # Get new settings to reset them

for q in compiler.chip_topology.qubits:
    settings.get_gate_node_for_locus("measure_fidelity", q).acquisition_type = "complex"

measure_settings = settings.get_gate_node_for_locus("measure_fidelity", [qubit])
measure_settings.amplitude_i = # Change the calibrated amplitude

job_definition, context = compiler.compile(circuits, settings=settings)
job = p.submit_playlist(job_definition, context=context)
job.wait_for_completion()

results = job.result(compiler).dataset[f"{qubit}__M_readout_single_shot"]

state_0_results = results.sel(circuit_index=0)
state_1_results = results.sel(circuit_index=1)
plt.figure()
plt.scatter(np.real(state_0_results), np.imag(state_0_results), label="Prepare 0", s=4)
plt.scatter(np.real(state_1_results), np.imag(state_1_results), label="Prepare 1", s=4)
plt.xlabel('Re')
plt.ylabel('Im')
plt.gca().set_aspect('equal')
plt.grid()
plt.legend();

We can see that for small values of the amplitude, the clusters overlap more and they are not well separated.

For high values of the amplitude instead, the clusters will be further apart, but they could be distorted. In fact, too high amplitudes could blast the readout resonator and the qubit into higher states, making other clusters appear in the plot above.

# Part 4 - Homework: Dispersive shift
The measurement of qubits in a quantum computer is needed to know anything about any experiment.

In this notebook, you will ...
* ... understand how the measurement signal depends on the state of the qubits.

By the end of this notebook, you will not only have a basic understanding of how to execute pulse schedules, but you will also know how to perform *sweeps* in Pulla.

In order to get started, make sure you have the appropriate packages installed:

## 1. Preparation

### 1.1 Connecting to the QPU station control
In the following, we will use the Pulla (Pulse Level Access) package. You can find the documentation [here](https://docs.iqm.tech/4.6/iqm-pulla/index.html
).

As a first step, we need to create a **Pulla object**. Conceptually, this is an IQM quantum computer client for connecting to the IQM server and constructing a circuit-to-pulse compiler. A compiler object defines the specific circuit-to-pulse compilation logic; it contains information about the quantum computer, like chip topology, the set of available native operations and other details that can be found [here](https://docs.iqm.tech/4.6/iqm-pulla/Configuration%20and%20Usage.html).

By default,  the `get_standard_compiler()` function fetches the default calibration set from the server.

Make sure you have the correct url and token and run the cell below:

In [ ]:
from iqm.pulla.pulla import Pulla

p = Pulla("https://resonance.iqm.tech", quantum_computer="garnet", token=input("Enter your Resonance API token: "))

compiler = p.get_standard_compiler()

## 2. Measurement operation

We have seen in the readout lab that qubit readout in superconducting systems is done via **superconducting resonators** coupled to the qubits in the QPU. More precisely, we perform a measurement by picking a frequency and looking at the signal we get from the readout resonator. The interesting fact is that, even if we are not sending a signal directly to the qubit, we can still infer the state of the qubit, as the resonator signal changes depending on it!

Our goal for this lab is then to prepare the qubit in different states and measure the resonator signal as a function of frequency. What we will observe is a **shift** in the resonator's frequency caused by the coupling with the qubit. This shift is typically a few MHz and the frequencies are in the microwave range. Let's start!

We perform two experiments:
1. Prepare a qubit in the $|0\rangle$ state and measure.
2. Prepare a qubit in the $|1\rangle$ state (applying an X gate) and measure.

Below we use the IQM Pulse syntax to define the circuits (you can find the documentation [here](https://docs.meetiqm.com/iqm-pulse/)). We will give a name to the circuits and append the necessary `CircuitOperation` to prepare the states above.

The PRX gate is defined as $R_{\phi}(\theta) = e^{-i(X cos(\phi)+Y sin(\phi))\frac{\theta}{2}}$. We can prepare the qubit in the $|1\rangle$ state above changing the rotation angle $\theta$ and setting the phase angle $\phi$ to zero.

We also include the measurement operation in each circuit; `measure_fidelity` is the destructive version of the regular `measure` operation, maximizing the readout fidelity and it is used at the end of the circuit.

In [ ]:
from iqm.pulse import Circuit, CircuitOperation as Op
import numpy as np

qubit = "QB1"
c0 = Circuit("0", [
    Op("measure_fidelity", (qubit,), args={"key": "M"})
])
c1 = Circuit("1", [
    Op("prx", (qubit,), args={"angle": , "phase": }), #TODO: set the right angles for preparing the qubit in state 1
    Op("measure_fidelity", (qubit,), args={"key": "M"})
])
circuits = [c0, c1]

### 2.1 Modify the calibration settings with sweeps

To see the **dispersive shift effect**, we need to repeat the preparation and measurement protocol for different readout frequencies. This can be done by looking at the calibrated frequency and spanning a range of frequencies around this reference value. In Pulla, performing a loop over a `settings` value is achieved by using the `sweep` method:

In [ ]:
settings = compiler.get_settings(circuits)
f_0 = settings.get_gate_node_for_locus("measure_fidelity", qubit).frequency
f_axis = f_0.sweep(np.linspace() + f_0.value)  # set a range of frequencies in Hz and a number of steps

We can now compile the circuits and execute the jobs corresponding to the different frequency values:

In [ ]:
job_definition, context = compiler.compile(circuits=circuits, settings=settings, sweeps=[f_axis])

job = # Submit the job for execution
job.wait_for_completion()
result = job.result(compiler)

## 3. Analyze the results

Now that the 'playlist' of instructions above has successfully run, we can extract the results and look at the readout signal as a function of frequency for the two different states. We can access the results with the key `M`, that was defined in the circuits above.

In [ ]:
import matplotlib.pyplot as plt

results = result.dataset[f"{qubit}__M_readout"]

state_0_results = #TODO
state_1_results = #TODO

plt.figure()
plt.plot(np.array(f_axis.data)/1e9, np.abs(state_0_results), label="Prepare 0")
plt.plot(np.array(f_axis.data)/1e9, np.abs(state_1_results), label="Prepare 1")
plt.xlabel("Readout frequency (GHz)")
plt.ylabel("Signal magnitude (a.u.)")
plt.legend();

We observe the dispersive shift effect: the qubit state shifts the spectrum of the resonator!

Now you know what happens behind the scenes everytime you append a measurement opeation to your circuit! We usually only monitor a fixed frequency and observe the change in magnitude and phase of the signal to return the result of a measurement.

In [ ]:
# Copyright 2024 IQM Quantum Computers (Hermanni Heimonen, Stefan Seegerer, Stefan Pogorzalek, Nadia Milazzo, Joni Ikonen)
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.